<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR NB08 &mdash; Luminesce Showcase Queries</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Ready-to-run Luminesce queries over the enriched security master, trade blotter, holdings, portfolio summary and F&O lifecycle events.</div>
</div>

<sub>IBOR pack sequence: NB00 &nbsp;&rarr;&nbsp; NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07 &nbsp;&rarr;&nbsp; **NB08**</sub>

# NB08: Luminesce Showcase Queries

Ready to use queries that demonstrate the rich security master and trade lifecycle data.

Copy any query below into **Data Virtualisation → Query Editor** to run it.

Scope: `ibor-training-v8`

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', 'lumipy', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import json
import lumipy as lp

with open("secrets.json") as f: secrets = json.load(f)
pat = secrets["api"]["accessToken"]
lumi = lp.get_client(access_token=pat, api_url=secrets["api"]["lusidUrl"].replace("/api", "/honeycomb"))

SCOPE = 'ibor-training-v8'
print("Connected to Luminesce")

---
## Query 1: Equity Security Master

Shows all equity instruments with identifiers, GICS classification, ESG rating, and fundamental data.

In [ ]:
q1 = """
SELECT i.DisplayName as [Name], i.ClientInternal, p2.Value as [Sector], p3.Value as [Country],
    p4.Value as [ESG Rating], p5.Value as [Market Cap], p6.Value as [Beta], p7.Value as [P/E Ratio],
    p8.Value as [Settlement Cycle], p9.Value as [Index Membership], i.LusidInstrumentId as [LUID]
FROM Lusid.Instrument i
    LEFT JOIN Lusid.Instrument.Property p2 ON p2.InstrumentScope='ibor-training-v8' AND p2.InstrumentIdType='LusidInstrumentId' AND p2.PropertyCode='Sector' AND p2.InstrumentId=i.LusidInstrumentId
    LEFT JOIN Lusid.Instrument.Property p3 ON p3.InstrumentScope='ibor-training-v8' AND p3.InstrumentIdType='LusidInstrumentId' AND p3.PropertyCode='Country' AND p3.InstrumentId=i.LusidInstrumentId
    LEFT JOIN Lusid.Instrument.Property p4 ON p4.InstrumentScope='ibor-training-v8' AND p4.InstrumentIdType='LusidInstrumentId' AND p4.PropertyCode='ESGRating' AND p4.InstrumentId=i.LusidInstrumentId
    LEFT JOIN Lusid.Instrument.Property p5 ON p5.InstrumentScope='ibor-training-v8' AND p5.InstrumentIdType='LusidInstrumentId' AND p5.PropertyCode='MarketCap' AND p5.InstrumentId=i.LusidInstrumentId
    LEFT JOIN Lusid.Instrument.Property p6 ON p6.InstrumentScope='ibor-training-v8' AND p6.InstrumentIdType='LusidInstrumentId' AND p6.PropertyCode='Beta' AND p6.InstrumentId=i.LusidInstrumentId
    LEFT JOIN Lusid.Instrument.Property p7 ON p7.InstrumentScope='ibor-training-v8' AND p7.InstrumentIdType='LusidInstrumentId' AND p7.PropertyCode='PriceEarningsRatio' AND p7.InstrumentId=i.LusidInstrumentId
    LEFT JOIN Lusid.Instrument.Property p8 ON p8.InstrumentScope='ibor-training-v8' AND p8.InstrumentIdType='LusidInstrumentId' AND p8.PropertyCode='SettlementCycle' AND p8.InstrumentId=i.LusidInstrumentId
    LEFT JOIN Lusid.Instrument.Property p9 ON p9.InstrumentScope='ibor-training-v8' AND p9.InstrumentIdType='LusidInstrumentId' AND p9.PropertyCode='IndexMembership' AND p9.InstrumentId=i.LusidInstrumentId
WHERE i.Scope='ibor-training-v8' AND i.Type='Equity' ORDER BY i.DisplayName LIMIT 50
"""

df = lumi.run(q1, quiet=True)
print(f"Equity Security Master: {len(df)} instruments, {len(df.columns)} columns")
display(df.head(20))

---
## Query 2: Trade Blotter with Lifecycle Fields

Shows transactions with counterparty, venue, settlement details, and commission.

In [ ]:
q2 = """
SELECT t.PortfolioCode as [Portfolio], t.TxnId as [Trade ID], t.Type, t.ClientInternal as [Instrument],
    t.Units, t.TradePrice as [Price], t.TotalConsideration as [Consideration], t.TradeCurrency as [CCY],
    t.TransactionDate as [Trade Date], t.SettlementDate as [Settle Date],
    p1.Value as [Counterparty], p2.Value as [Venue], p3.Value as [Broker],
    p4.Value as [Settlement Instruction], p5.Value as [Commission], p6.Value as [Trade Source], p7.Value as [Strategy]
FROM Lusid.Portfolio.Txn t
    LEFT JOIN Lusid.Portfolio.Txn.Property p1 ON p1.PortfolioScope='ibor-training-v8' AND p1.PortfolioCode=t.PortfolioCode AND p1.TxnId=t.TxnId AND p1.PropertyCode='CounterpartyId'
    LEFT JOIN Lusid.Portfolio.Txn.Property p2 ON p2.PortfolioScope='ibor-training-v8' AND p2.PortfolioCode=t.PortfolioCode AND p2.TxnId=t.TxnId AND p2.PropertyCode='ExecutionVenue'
    LEFT JOIN Lusid.Portfolio.Txn.Property p3 ON p3.PortfolioScope='ibor-training-v8' AND p3.PortfolioCode=t.PortfolioCode AND p3.TxnId=t.TxnId AND p3.PropertyCode='Broker'
    LEFT JOIN Lusid.Portfolio.Txn.Property p4 ON p4.PortfolioScope='ibor-training-v8' AND p4.PortfolioCode=t.PortfolioCode AND p4.TxnId=t.TxnId AND p4.PropertyCode='SettlementInstructionType'
    LEFT JOIN Lusid.Portfolio.Txn.Property p5 ON p5.PortfolioScope='ibor-training-v8' AND p5.PortfolioCode=t.PortfolioCode AND p5.TxnId=t.TxnId AND p5.PropertyCode='CommissionAmount'
    LEFT JOIN Lusid.Portfolio.Txn.Property p6 ON p6.PortfolioScope='ibor-training-v8' AND p6.PortfolioCode=t.PortfolioCode AND p6.TxnId=t.TxnId AND p6.PropertyCode='TradeSource'
    LEFT JOIN Lusid.Portfolio.Txn.Property p7 ON p7.PortfolioScope='ibor-training-v8' AND p7.PortfolioCode=t.PortfolioCode AND p7.TxnId=t.TxnId AND p7.PropertyCode='Strategy'
WHERE t.PortfolioScope='ibor-training-v8' AND t.PortfolioCode='IBOR-EQ' ORDER BY t.TransactionDate DESC LIMIT 30
"""

df = lumi.run(q2, quiet=True)
print(f"Trade Blotter (IBOR-EQ): {len(df)} trades, {len(df.columns)} columns")
display(df.head(15))

---
## Query 3: Enriched Holdings Report

Joins holdings with instrument properties for a position report with sector, country, and ESG data.

In [ ]:
q3 = """
SELECT h.PortfolioCode as [Portfolio], i.DisplayName as [Instrument], h.Units, h.CostAmount as [Cost Basis],
    h.CostCurrency as [Currency], p1.Value as [Sector], p2.Value as [Country], p3.Value as [ESG Rating],
    t1.Value as [Strategy], t2.Value as [Custodian]
FROM Lusid.Portfolio.Holding h
    LEFT JOIN Lusid.Instrument i ON i.LusidInstrumentId=h.LusidInstrumentId AND i.Scope='ibor-training-v8'
    LEFT JOIN Lusid.Instrument.Property p1 ON p1.InstrumentScope='ibor-training-v8' AND p1.InstrumentIdType='LusidInstrumentId' AND p1.PropertyCode='Sector' AND p1.InstrumentId=h.LusidInstrumentId
    LEFT JOIN Lusid.Instrument.Property p2 ON p2.InstrumentScope='ibor-training-v8' AND p2.InstrumentIdType='LusidInstrumentId' AND p2.PropertyCode='Country' AND p2.InstrumentId=h.LusidInstrumentId
    LEFT JOIN Lusid.Instrument.Property p3 ON p3.InstrumentScope='ibor-training-v8' AND p3.InstrumentIdType='LusidInstrumentId' AND p3.PropertyCode='ESGRating' AND p3.InstrumentId=h.LusidInstrumentId
    LEFT JOIN Lusid.Portfolio.Holding.Property t1 ON t1.PortfolioScope='ibor-training-v8' AND t1.PortfolioCode=h.PortfolioCode AND t1.PropertyScope='ibor-training-v8' AND t1.PropertyCode='Strategy' AND t1.LusidInstrumentId=h.LusidInstrumentId AND t1.SubHoldingKey=h.SubHoldingKey
    LEFT JOIN Lusid.Portfolio.Holding.Property t2 ON t2.PortfolioScope='ibor-training-v8' AND t2.PortfolioCode=h.PortfolioCode AND t2.PropertyScope='ibor-training-v8' AND t2.PropertyCode='CustodianAccount' AND t2.LusidInstrumentId=h.LusidInstrumentId AND t2.SubHoldingKey=h.SubHoldingKey
WHERE h.PortfolioScope='ibor-training-v8' AND h.PortfolioCode='IBOR-SP500' AND h.EffectiveAt='2024-09-30' AND h.HoldingType='Position' ORDER BY h.CostAmount DESC LIMIT 30
"""

df = lumi.run(q3, quiet=True)
print(f"Enriched Holdings (IBOR-SP500): {len(df)} positions")
display(df.head(15))

---
## Query 4: Cross Portfolio Summary

Aggregated view across all portfolios showing position count and total cost.

In [ ]:
q4 = """
SELECT h.PortfolioCode as [Portfolio], COUNT(*) as [Positions], SUM(h.CostAmount) as [Total Cost], h.CostCurrency as [CCY]
FROM Lusid.Portfolio.Holding h
WHERE h.PortfolioScope='ibor-training-v8' AND h.EffectiveAt='2024-09-30' AND h.HoldingType='Position'
GROUP BY h.PortfolioCode, h.CostCurrency ORDER BY SUM(h.CostAmount) DESC
"""

df = lumi.run(q4, quiet=True)
print(f"Portfolio Summary: {len(df)} rows")
display(df)

---
## Query 5: F&O Lifecycle Events

Shows the futures and options expiry/settlement transactions.

In [ ]:
q5 = """
SELECT t.TxnId as [Event ID], t.Type as [Event Type], t.ClientInternal as [Instrument], t.Units,
    t.TradePrice as [Settlement Price], t.TotalConsideration as [P&L], t.TransactionDate as [Date]
FROM Lusid.Portfolio.Txn t
WHERE t.PortfolioScope='ibor-training-v8' AND t.PortfolioCode='IBOR-MA'
    AND t.Type IN ('FutureCashSettlement','CashSettledOptionExercise','Expiry')
ORDER BY t.TransactionDate
"""

df = lumi.run(q5, quiet=True)
print(f"F&O Lifecycle Events: {len(df)} events")
display(df)

---
## Summary

These queries can be copied into **Data Virtualisation → Query Editor** and saved via the **Save** button for reuse.

Key patterns demonstrated:

1. **Property joins**: `LEFT JOIN Lusid.Instrument.Property` to enrich instruments with identifiers, GICS, ESG
2. **Transaction property joins**: `LEFT JOIN Lusid.Portfolio.Txn.Property` to add counterparty, venue, commission
3. **Holding property joins**: `LEFT JOIN Lusid.Portfolio.Holding.Property` for SHK based strategy/custodian
4. **Aggregation**: `GROUP BY` and `SUM` for portfolio level summaries
5. **Filtering**: `WHERE` on specific portfolios, dates, holding types, transaction types